In [0]:
import re
import os
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
import logging
logger = logging.getLogger("modelling")


In [0]:
def ptab(df, first_ptab):
    start = True
    df['PTAB'] = np.nan
    for idx, row in df.iterrows():
        if start and idx>0:
            df['PTAB'].iloc[idx] = first_ptab
        if row['sourceID'] == "MRI_FRR_257":
            ptab = row['text'].split()[-1]  # position appears on the last position of the string
            df['PTAB'].iloc[idx] = ptab
            start = False
    return df

In [0]:
def modelling(serial_number):
    print(f"Serial: {serial_number} Data Processing Started")
    eventlog_df = spark.read.table("hive_metastore.eventlog.common_eventlog").select(
            F.date_format(F.col("EventDateTime"), "yyyy-MM-dd HH:mm:ss").alias("EventDateTime"),
            "SerialNumber",
            "MessageIdentification",
            "TimeZoneOffset",
            "Message",
            "Line",
        )

    result_df = (
        eventlog_df
        .filter(F.col("SerialNumber") == serial_number)
        .withColumn(
            "AdjustedEventDateTime", 
            F.timestamp_seconds(F.unix_timestamp(F.col("EventDateTime")) + F.col("TimeZoneOffset"))
        )
        .filter(F.to_date(F.col("AdjustedEventDateTime")).between(start_date, end_date))
        .filter(
            (F.col("MessageIdentification").startswith("MRI_MSR")) |
            (F.col("MessageIdentification").isin([
                "MRI_MSR", "MRI_FRR_18", "MRI_EXU_95", "MRI_EXU_89", 
                "MRI_CCS_1008", "MRI_MPT_1005", "MRI_SUT_1005",
                "MRI_FRR_257"  # PTAB events
            ]))
        )
        .select(
            F.date_format(F.col("AdjustedEventDateTime"), "yyyy-MM-dd HH:mm:ss").alias("AdjustedEventDateTime"),
            "MessageIdentification",
            "Message",
            "Line",
        )
        .orderBy(F.col("AdjustedEventDateTime").asc())
    )

    result_df = result_df.toDF("datetime", "sourceID", "text", "Line")
    pandas_df = result_df.toPandas()

    # Extract PTAB (patient table position) from MRI_FRR_257 events
    pandas_df = ptab(pandas_df, 0)
    pandas_df['PTAB'] = pandas_df['PTAB'].ffill().bfill()

    df_joint = pd.DataFrame()
    joint_coilList = []

    df_ini, coilList = data_transformation(pandas_df)
    joint_coilList.extend(coilList)
    df_select = df_ini[(df_ini['sourceID'].isin(['MRI_MSR_104', 'MRI_MSR_34', 'MRI_MSR_26', 'MRI_MSR_22', 'MRI_MSR_40', 'MRI_MSR_25', 'MRI_MSR_24'])) & (df_ini['FinishEvent'] != False)] 
    df_pre = df_select.copy()
    df_pre.drop(columns=["sourceID","text"], axis = 1, inplace = True)

    df_sorted = measurement_time(df_pre)
    df_sorted.fillna(False, inplace=True)
    df_sorted['SN'] = serial_number
    df_joint = pd.concat([df_joint,df_sorted])

    df_joint['FinishEvent'].fillna(False, inplace=True)
    df_joint = df_joint[df_joint["FinishEvent"] != False]
    df = order_cols(df_joint, joint_coilList)

    # Restore PTAB column (not included by order_cols)
    if 'PTAB' in df_joint.columns:
        df['PTAB'] = df_joint['PTAB'].values

    path = r"/dbfs/FileStore/pat_exam"
    os.chdir(path)

    s1 = df['PatientID'].replace({'False': np.nan})
    s1b = df['BodyPart'].replace({'False': np.nan})
    mask1 = (df['PatientID'] == 'False') & ((df['Sequence'].str.contains('Scout', case=False)) | (df['Protocol'].str.contains('localizer', case=False)) | (df['Protocol'].str.contains('Scout', case=False)))
    s2 = s1.bfill()
    s2b = s1b.bfill()
    df.loc[mask1, 'PatientID'] = s2
    df.loc[mask1, 'BodyPart'] = s2b

    s3 = s1.ffill()
    s3b = s1b.ffill()
    mask2 = s3.eq(s1.bfill())
    df.loc[mask2, 'PatientID'] = s3
    df.loc[mask2, 'BodyPart'] = s3b

    df['PatientID'] = df['PatientID'].fillna(method='ffill')
    df['BodyPart'] = df['BodyPart'].fillna(method='ffill')

    mask3 = (df['PatientID'] == 'False') & (df['ConnectedCoils'] == df['ConnectedCoils'].shift())
    df.loc[mask3, 'PatientID'] = df['PatientID'].shift()
    df.loc[mask3, 'BodyPart'] = df['BodyPart'].shift()

    df['startTime'] = pd.to_datetime(df['startTime'])
    df['endTime'] = pd.to_datetime(df['endTime'])

    df['PatientID'] = df['PatientID'].fillna(method='ffill')
    df['BodyPart'] = df['BodyPart'].fillna(method='ffill')

    mask4 = (df['PatientID'] == 'False') & ((df['endTime'].shift(-1) - df['startTime'])<= pd.Timedelta(minutes=3))
    df.loc[mask4, 'PatientID'] = df['PatientID'].shift(-1)
    df.loc[mask4, 'BodyPart'] = df['BodyPart'].shift(-1)

    mask5 = (df['duration'] == 0) & (df['FinishEvent'] == 'Successful')
    df = df[~mask5]

    df_drop = df.drop(df[df['duration'] > 4000].index)
    df_city = match_country(df,df_cube)
    df_fin = match_bp(df_city,df_bp)

    ### ADD PAUSE COLUMN ###
    df_fin['pauseTime'] = (df_fin['startTime'].shift(-1) - df_fin['endTime']).dt.total_seconds()
    df_fin['StepCount'] = df_fin.groupby('PatientID').cumcount() + 1
    df_fin.loc[df_fin['PatientID'] != df_fin['PatientID'].shift(-1), 'pauseTime'] = 0

    ### ADD EXAMINATION DATA (Age, Weight, Height, Direction) ###
    try:
        sd = pd.to_datetime(start_date)
        ed = pd.to_datetime(end_date)
        exam_df = spark.read.table("hive_metastore.examination.examination_workflow")
        exam_filtered = (
            exam_df
            .filter(F.col("SerialNumber") == int(serial_number))
            .filter(
                (F.col("Year") * 10000 + F.col("Month") * 100 + F.col("Day"))
                .between(int(start_date.replace("-", "")), int(end_date.replace("-", "")))
            )
            .withColumn("PatientId", F.col("WorkflowValues")["PatientId"])
            .withColumn("Age", F.col("WorkflowValues")["Age"])
            .withColumn("Weight", F.col("WorkflowValues")["Weight"])
            .withColumn("Height", F.col("WorkflowValues")["Height"])
            .withColumn("Direction", F.col("WorkflowValues")["Direction"])
            .select("PatientId", "Age", "Weight", "Height", "Direction")
            .dropDuplicates(["PatientId"])
        )
        exam_pdf = exam_filtered.toPandas()
        df_fin = df_fin.merge(exam_pdf, left_on='PatientID', right_on='PatientId', how='left')
        if 'PatientId' in df_fin.columns:
            df_fin.drop(columns=['PatientId'], inplace=True)
        print(f"Serial: {serial_number} Examination data merged ({len(exam_pdf)} patients)")
    except Exception as e:
        print(f"Warning: Could not merge examination data for {serial_number}: {e}")
        df_fin['Age'] = np.nan
        df_fin['Weight'] = np.nan
        df_fin['Height'] = np.nan
        df_fin['Direction'] = np.nan

    # Save dataset
    csv_name = f'/dbfs/FileStore/pat_exam/DATA_{serial_number}.csv'
    df_fin.to_csv(csv_name, index = False, header = True)
    print(f"Serial: {serial_number} Data Saved")

    return "Success"

In [0]:


def expand_coils(coils, group):
    """From extracted coils, this function creates a list with precise ID of the coils being used.

    Args:
        coils (str): string directly extracted from the text cell. Example: 'HC1-4,NC1,2'
        group (str): string containing the group of the coil, usually 0 or 1

    Return:
        list: list containing the expanded coils used in the measurement.
        Example: ['#0 HC1','#0 HC2','#0 HC3','#0 HC4','#0 NC1','#0 NC2']
    """
    print("expand_coils Started")

    connected_coil = coils.split(',')
    sc = []
    for i,elem in enumerate(connected_coil):
        if elem:
            if '.' in elem or elem[-1]=='-':    # Eliminate possible errors at the end of the string
                elem=elem[:-1]        
            if not elem.isnumeric():
                elem = elem.strip()     # Remove spaces at the end and beginning of the string
                coil = elem[:2]         # Coils are always formed by 2 letters and 1 digit. Select the letters
                if '-' in elem:         # Check if the coils are consecutives in a list (1-4)
                    st = elem[2:]       # Select only digits
                    st_sp = st.split('-')       # Take the digits of beginning and end of the list
                    if len(st_sp)>1:            # Only to avoid errors, check that there are 2 digits in the list
                        try:
                            for i in range(int(st_sp[0]),int(st_sp[1])+1):
                                sc.append('#'+group+'_'+coil+str(i))        # Create list of coils between the two previous numbers
                        except:
                            pass
                else:
                    sc.append('#'+group+'_'+elem)       # If there is no list of consecutive coils, write directly the coil complete name
            else:
                con = '#'+group+'_'+coil+elem           # Write directly the coil when it is individually written in the list
                sc.append(con)
    print("expand_coils Ended")
    return sc       # Return list of all coils for a measurement

def get_seq(text):
    """Get sequence and protocol of a measurement

    Args:
        text (str): text cell of a pandas dataframe's row

    Returns:
        str: sequence name
        str: protocol name
    """
    print("get_seq Started")

    sequence_type = re.search("Sequence: '(.*)'", text).group(1)       # Capture sequence, including model (tse, gre...)
    protocol = re.search("Protocol: '(.*)',",text).group(1)

    print("get_seq Ended")
    return sequence_type, protocol

def get_body_patient(text):
    """This function extracts info from the MRI_EXU_95 event, specifically body
    part and anonymised patient ID of a measurement.

    Args:
        text (string): text in the row of the dataframe

    Returns:
        body (string): body part being examined in the measurement
        patient_id (string): anonymised patient ID of the patient being examined
        flag_patient (bool): patient was found or not
    """
    print("get_body_patient Started")

    body = re.search("with body part < (.*) >", text).group(1)
    if ' ' in body:     # Sometimes an extra <> appears, with info about contrast agent <body> <contrast>. We want only the body part.
        try:
            body = re.search("with body part < (.*) > <", text).group(1)
        except:
            body = np.NaN
        try:
            patient_id = re.search("Anonymised Patient ID < (.*) >", text).group(1)
            flag_patient = True
        except:
            patient_id = np.NaN
            flag_patient = False
    
    print("get_body_patient Ended")
    return body, patient_id, flag_patient
  
def data_transformation(df_):
    """This function takes a eventlog in pandas dataframe format and outputs a dataframe with the features that are needed for datamodelling:
    Binary columns: Coil elements, DOT
    Categorical columns: body part, body part group, sequence, FinishEvent, Serial Number, anonymized Patient ID, Country, System type
    Timestamp columns: start time, end time (date and time of the day)
    Numerical column: duration of the measurement in seconds

    Args:
        df_ (dataframe): dataframe with events, output of read_eventlog function

    Returns:
        dataframe: features for datamodelling
        list: list containing the different coil elements used by the customer
    """
    print("data_transformation Started")

    # Initializaton of variables
    sc = []                         # Create an empty list to hold coils
    start = False                   # Set a flag to indicate whether a measurement has started
    coilList = []                   # Create an empty list to hold the names of unique coils
    df_ = df_[~df_['text'].str.contains('AdjustSeq|ServiceSeq', case = False)]
    df_ = df_.reset_index(drop=True)    # Keep the indexes up to date after dropping the adjust and service measurements
    df = pd.DataFrame(index=range(df_.shape[0]), columns=["Sequence"])    # Create a new dataframe to hold new extracted information
    df["Sequence"] = "False"        # Set the sequence for each row to "False"
    flag_dot = False                # Flag for DOT measurement
    conCoils = ''                   # Initialization of string to save the connected coils
    start_index = 0                 # Initialization to save index of a measurement start
    finishEvents = ['MRI_MSR_104', 'MRI_MSR_18', 'MRI_FRR_18','MRI_MSR_21', 'MRI_MSR_34', 'MRI_MSR_26', 'MRI_MSR_22', 'MRI_MSR_40', 'MRI_MSR_25', 'MRI_MSR_24']
    finishEventsReduced = ['MRI_MSR_104', 'MRI_MSR_34', 'MRI_MSR_26', 'MRI_MSR_22', 'MRI_MSR_40', 'MRI_MSR_25', 'MRI_MSR_24']
    outEXU = False                  # Initialization of bool variable. It will be true if EXU event is found outside the measurement boundaries

    for i, row in df_.iterrows():

        if row.sourceID == 'MRI_EXU_89':    # Event for DOT measurement
            flag_dot = True
        if row.sourceID == 'MRI_CCS_1008':  # Event with ConnectedCoils info
            conCoils = ''
            coil_ids = re.findall(r"CoilID\s+'(\w+)'", row.text)
            conCoils = "-".join(coil_ids)

        if row.sourceID == 'MRI_MSR_100':   # Event with sequence-protocol info
            start = True        # Previously dropped all Service and Adjustment sequences
            try:
                seq, protocol = get_seq(row.text)
            except:
                continue
            datetime = row.datetime     # Save datetime of measurement start
            # We don't need a flag_seq because the sequence is always going to be there, as it is itself the flag to start a measurement
            start_index_prev = start_index
            start_index = i


        if start and row.sourceID == 'MRI_EXU_95':      # Event containing PatientID and BodyPart
            body, patient_id, flag_patient = get_body_patient(row.text)
            startEXU = True     # Record that MRI_EXU_95 was found after MRI_MSR_100

        if start and row.sourceID == 'MRI_MSR_101':     # Event containing coil elements
            s = row.text
            group = re.search("#(\d):", s).group(1)
            selected_coil = re.search(": '(.*)'\)", s).group(1)
            # There are cases where the coils elements come in different messages (#0 or #1). If sc is not empty, do not overwrite the info but append it
            if not sc:
                sc = expand_coils(selected_coil,group)
            else:
                for element in expand_coils(selected_coil,group):
                    sc.append(element)
        
        if row.sourceID in finishEvents:
            if start and row.sourceID in finishEventsReduced:   # Save info everytime a measurement is finished
                df_.at[i,'startTime'] = datetime
                # Define selected coil elements
                for elem in sc:
                    if elem not in df_.columns:     # If it is a new coil element, store it in a list
                        coilList.append(elem)
                    df_.at[i,elem] = True           # One binary column per coil

                # Define Sequence and Protocol columns
                df.at[i,"Sequence"] = seq
                df.at[i,"Protocol"] = protocol

                # Define FinishEvent column
                if row.sourceID == 'MRI_MSR_104':
                    df.at[i,"FinishEvent"] = 'Successful'
                elif row.sourceID == 'MRI_MSR_22':
                    df.at[i,"FinishEvent"] = 'Start MeasSys Failed'
                elif row.sourceID == 'MRI_MSR_24':
                    df.at[i,"FinishEvent"] = 'Stopped by Scanner'
                elif row.sourceID == 'MRI_MSR_25':
                    df.at[i,"FinishEvent"] = 'Stopped by ImageR'
                elif row.sourceID == 'MRI_MSR_26':
                    df.at[i,"FinishEvent"] = 'Scanning Error'
                elif row.sourceID == 'MRI_MSR_34':
                    df.at[i,"FinishEvent"] = 'Stopped by User'
                elif row.sourceID == 'MRI_MSR_40':
                    df.at[i,"FinishEvent"] = 'Stopped by CoilChange'

                # Define BodyPart column
                if not startEXU:            # If MRI_EXU_95 event was not found after the start of the measurement (MRI_MSR_100)
                    # Check 2 lines above if there is a MRI_EXU_95 Event
                    if df_['sourceID'].iloc[start_index-2] == 'MRI_EXU_95':
                        outEXU = True               # MRI_EXU_95 event is found before start of the measurement
                        k = start_index-2           # Save the index of MRI_EXU_95
                    elif df_['sourceID'].iloc[start_index-1] == 'MRI_EXU_95':
                        outEXU = True
                        k = start_index-1
                    else:       # If event is missing but not found within the previous messages, check within time interval of 4 minutes with the last recorded MRI_EXU_95 event
                        df_date = pd.DataFrame()
                        # Convert the "datetime" column to a Pandas DateTime object
                        df_date['datetime'] = pd.to_datetime(df_['datetime'])
                        # Set the index to the "datetime" column
                        df_date.set_index('datetime', inplace=True)
                        current_row = df_date.iloc[start_index]             # Current MRI_EXU_95 event
                        previous_row = df_date.iloc[start_index_prev]       # Last recorded MRI_EXU_95 event
                        is_within_three_min = (current_row.name - previous_row.name) <= pd.Timedelta(minutes=4)
                        if is_within_three_min:
                            if body:        # Check that previously body was stored, this body part will be the same for the current measurement
                                startEXU = True     # Activate flag that MRI_EXU_95 event was found. In this case: info from previous is being used
                    if outEXU:      # If MRI_EXU_95 event was found before start measurement, extract info from the message
                        body, patient_id, flag_patient = get_body_patient(df_.iloc[k].text)
                        startEXU = True     # Activate flag that MRI_EXU_95 event was found     
                        outEXU = False      # Reset bool for out of boundaries MRI_EXU_95 event
                if startEXU:        # If MRI_EXU_95 was found -> info extracted, store the info about body part and patient in columns
                    df_.at[i,'BodyPart'] = body
                    if flag_patient:
                        df_.at[i,'PatientID'] = patient_id
                    else:
                        df_.at[i,'MissingPatientID'] = True
                else:
                    df_.at[i,'MissingBodyPart'] = True
                    df_.at[i,'MissingPatientID'] = True

                if bool(conCoils):      # Check that connected coils were extracted
                    df_.at[i,'ConnectedCoils'] = conCoils
                else:
                    df_.at[i,'MissingConCoils'] = True
                
                if flag_dot:
                    df_.at[i,'DOT'] = True
                else:
                    df_.at[i,'DOT'] = False

                df.fillna(False, inplace=True)      # Set to FALSE every missing information

            # Reset variables
            start = False
            startEXU = False
            sc = []
        if row.sourceID == "MRI_MPT_1005":  # Set DOT workflow False when a new patient is registered
            flag_dot = False
            
    df_out = pd.concat([df_, df], axis=1)

    print("data_transformation Ended")
    return df_out, coilList

def order_cols(df, joint_coilList):
    """This function orders the columns of a dataframe after all the customers are analyzed and their features have been extracted.

    Args:
        df (dataframe): dataframe with features for datamodelling.
        joint_coilList (list): list containing all coils used by all the customers being modelled.

    Returns:
        dataframe: dataframe with ordered columns, first all binary columns of coil elements, then the rest.
    """
    print("order_cols Started")

    # Order columns of output dataframe
    df_out = pd.DataFrame()
    joint_coilList = list(set(joint_coilList))

    existing_coils = [c for c in joint_coilList if c in df.columns]
    move_cols_coils = pd.concat([df.pop(x) for x in existing_coils], axis=1)



    #move_cols_coils = pd.concat([df.pop(x) for x in joint_coilList], axis=1)      # Extract columns that are in the middle of the DataFrame
    #df_out = pd.concat([df_out, move_cols_coils], 1)       # Move these columns at the end of the DataFrame
    df_out = pd.concat([df_out, move_cols_coils], axis=1)
    move_body = df.pop('BodyPart')
    move_seq = df.pop('Sequence')
    move_concoils = df.pop('ConnectedCoils')
    move_dot = df.pop('DOT')
    move_id = df.pop('PatientID')
    move_prot = df.pop('Protocol')
    move_sn = df.pop('SN')
    move_fin = df.pop('FinishEvent')
    #df_out = pd.concat([df_out, move_concoils, move_body, move_seq, move_prot, move_dot, move_id, move_fin, move_sn], 1)
    df_out = pd.concat([df_out, move_concoils, move_body, move_seq, move_prot, move_dot, move_id, move_fin, move_sn], axis=1)
    if any(col.startswith('Missing') for col in df.columns):
        cols_missing = [col for col in df.columns if "Missing" in col]
        move_cols_missing = pd.concat([df.pop(x) for x in cols_missing], axis=1)      # Extract columns that are in the middle of the DataFrame
        df_out = pd.concat([df_out, move_cols_missing], axis=1)
    # Convert to integer before adding the datetime
    df_xlsx = df_out.fillna(False)
    # Add datetime
    move_starttime = df.pop('startTime')
    move_duration = df.pop('duration')
    move_endtime = df.pop('endTime')
    #df_xlsx = pd.concat([df_xlsx, move_starttime, move_endtime, move_duration.astype('int64')], 1)
    df_xlsx = pd.concat([df_xlsx, move_starttime, move_endtime, move_duration.astype('int64')], axis=1)

    print("order_cols Ended")
    return df_xlsx

def measurement_time(df):
    """This functions sorts measurements by datetime and creates the columns endTime and duration (seconds)

    Args:
        df (dataframe): dataframe including the features of the customer

    Returns:
        dataframe: dataframe with two new features, endTime and duration
    """

    #print("measurement_time Started")

    df['datetime'] = pd.to_datetime(df['datetime'])
    df_sorted = df.sort_values(by=['datetime'])
    df_sorted = df_sorted.drop(df_sorted[df_sorted['Sequence'] == 'False'].index)   # Drop rows where the sequence is False, that means that two measurements overlap and the first sequence is not considered. Consider only second measurement
    df_sorted.rename(columns={"datetime": "endTime"}, inplace = True)
    df_sorted['startTime'] = pd.to_datetime(df_sorted['startTime'])
    df_sorted['duration']  = (df_sorted['endTime'] - df_sorted['startTime']).dt.total_seconds()

    #print("measurement_time Ended")
    return df_sorted

def match_country(df_info, df_cube):
    """This function adds the country and system type information to each customer.

    Args:
        df_info (dataframe): it contains the extracted features of the customer
        df_cube (dataframe): it contains the country and system type information

    Returns:
        dataframe: it includes features + country and system type columns
    """

    #print("match_country Started")
    df_cube.rename(columns={"Serial": "SN"}, inplace = True)
    df_cube.drop_duplicates(subset='SN', keep='first', inplace=True)
    df_info['SN'] = df_info['SN'].astype(str)
    df_cube['SN'] = df_cube['SN'].astype(str)
    merged_df = pd.merge(df_info, df_cube, on='SN', how='inner')
    merged_df['Country'] = merged_df['Country'].fillna('Unknown')

    #print("match_country Ended")
    return merged_df

def match_bp(df_info, df_bp):
    """This function adds the body part group information to each measurement.

    Args:
        df_info (_type_): it contains the extracted features of the customer
        df_bp (_type_): it contains the body part group information. It is taken from the cube.

    Returns:
        dataframe: it includes features + body part group column
    """

    print("match_bp Started")
    df_info['bp_lower'] = df_info['BodyPart'].str.lower()
    df_bp['bp_lower'] = df_bp['BodyPart'].str.lower()
    df_bp.drop(columns='BodyPart', inplace=True)
    merged_df = pd.merge(df_info, df_bp, on='bp_lower', how='left')
    merged_df['BodyGroup'] = merged_df['BodyGroup'].fillna('Unknown')
    merged_df.drop(columns='bp_lower', inplace=True)

    print("match_bp Ended")
    return merged_df

In [0]:
pip install openpyxl

In [0]:
serial_list = ['183242', '176148', '176227', '175912', '175670', '183776', '182625', '176615', '176240', '175828', '183210', '183105', '176075', '177632', '175892', '186466', '183697', '183166', '177602', '175895', '175631', '202063', '183588', '183427', '182792', '176562', '176005', '175832', '183538', '182730', '202065', '183169', '175622', '175795', '183693', '183346', '175764', '183700', '182629', '202008', '186443', '186410', '175931', '175852', '183391', '202012', '183579', '175987', '186424', '177667', '177648', '176003', '183089', '176285', '175698', '183063', '176487', '183321', '202050', '183318', '176017', '176234', '186488', '183624', '176078', '175744', '183011', '176026', '183116', '186518', '183508', '176744', '183298', '183333', '182882', '175925', '186433', '176379', '202541', '176125', '182885', '182900', '183057', '183255', '182648', '175623', '202044', '183114', '182650', '176620', '183101', '182622', '183320', '202546', '186438', '183528', '175812', '202575', '175715', '182779', '188001', '183470', '183670', '182977', '176750', '202027', '183384', '202516', '183137', '183349', '176132', '175781', '182966', '183780', '183658', '175811', '202073', '182884', '186442', '182809', '182901', '175666', '183572', '183608', '182907', '202045', '175710', '182721', '175706', '176548', '183413', '176009', '182970', '175906', '177624', '183618', '175711', '176356', '183292', '176389', '182690', '186462', '183109', '183035', '175727', '176588', '183472', '202043', '175997', '182975', '183431', '182821', '183209', '183031', '176430', '175848', '183426', '175944', '183055', '176123', '202594', '186489', '182636', '186477', '176133', '177658', '186482', '175957', '183423', '175697', '175994', '175725', '183311', '176225', '183326', '183141', '175836', '177657', '182921', '176394', '175973', '177609', '175720', '188003', '176371', '182847', '202047', '176549', '183282', '202024', '176673', '202007', '202551', '183445', '186517', '186492', '183451', '176520', '176498', '175770', '183583', '183230', '175693', '183228', '176457', '202538', '186486', '176141', '183219', '175742', '183130', '175777', '175755', '175869', '176016', '183040', '176659', '175678', '176244', '182979', '182698', '176015', '202524', '182948', '176359', '177623', '182767', '177605', '175952', '182656', '175924', '183199', '202512', '182798', '183564', '183241', '183509', '176654', '183364', '182795', '176767', '176482', '183148', '183150', '183334', '183034', '183046', '183289', '182834', '176186', '176275', '183742', '202009', '175606', '202049', '175962']

'''
# SERIAL_PATH = "dbfs:/FileStore/pat_exam/TEST.csv"
SERIAL_PATH = "/dbfs/FileStore/test.csv"

# File is actually Excel (.xlsx) despite .csv extension — use pandas
try:
    serial_df_pd = pd.read_excel(SERIAL_PATH)
    serial_list = serial_df_pd['Serial'].astype(str).tolist()
    print(f"Loaded {len(serial_list)} serials from Excel file")
except Exception as e:
    print(f"Warning: Could not read serial file ({e}), using hardcoded list ({len(serial_list)} serials)")
'''    

start_date = "2025-04-05"
end_date = "2025-04-07"

df_cube = pd.read_csv(
    '/dbfs/FileStore/pat_exam/aprdataWorldreport_sumv2.csv',
    usecols=['Serial', 'Country', 'Systemtype']
)

df_bp = pd.read_excel(
    '/dbfs/FileStore/pat_exam/bodyupdated.xlsx'
)

In [0]:
# for serial in serial_list:
#     result = modelling(serial)


In [0]:
test_serial = '183242'
result = modelling(test_serial)
print(f"\nResult: {result}")
displayHTML(f'<a href="files/pat_exam/DATA_{test_serial}.csv">Download DATA_{test_serial}.csv</a>')

In [0]:
# dbutils.fs.cp("dbfs:/FileStore/pat_exam/DATA_183242.csv", "dbfs:/FileStore/pat_exam/COPY_OF_DATA_183242.csv")

In [0]:
fpath = "files/pat_exam/DATA_183242.csv"
displayHTML(f'<a href="files/pat_exam/DATA_183242.csv">Link to file</a>')

In [0]:
exam_df = spark.read.table("hive_metastore.examination.examination_workflow")
exam_filtered = (
    exam_df
    .filter(F.col("SerialNumber") == 183242)
    .filter(
        (F.col("Year") * 10000 + F.col("Month") * 100 + F.col("Day"))
        .between(int(start_date.replace("-", "")), int(end_date.replace("-", "")))
    )
    .withColumn("PatientId", F.col("WorkflowValues")["PatientId"])
    .withColumn("Age", F.col("WorkflowValues")["Age"])
    .withColumn("Weight", F.col("WorkflowValues")["Weight"])
    .withColumn("Height", F.col("WorkflowValues")["Height"])
    .withColumn("Direction", F.col("WorkflowValues")["Direction"])
    .select("PatientId", "Age", "Weight", "Height", "Direction")
    .dropDuplicates(["PatientId"])
)
print(f"Matching patients for serial 183242: {exam_filtered.count()}")
display(exam_filtered.limit(5))
